## Project 1

### Part A

In [ ]:
# Code part A
# Martin

In [ ]:
# Plots part A
# Martin

### Part B

In [ ]:
import pypsa
import pandas as pd
import numpy as np

# ----------------------------
# Create time snapshots
# ----------------------------
hours = pd.date_range("2025-01-01 00:00", periods=24, freq="h")

# Demand profile
load = pd.Series(100.0, index=hours)

# ----------------------------
# Weather year A
# ----------------------------
wind_cf_A = pd.Series(
[0.45,0.42,0.40,0.38,0.35,0.33,0.30,0.32,0.36,0.40,0.43,0.46,
0.48,0.50,0.47,0.44,0.41,0.39,0.37,0.36,0.38,0.41,0.44,0.46],
index=hours)

solar_cf_A = pd.Series(
[0,0,0,0,0,0.02,0.10,0.25,0.45,0.60,0.70,0.75,
0.72,0.65,0.50,0.30,0.12,0.02,0,0,0,0,0,0],
index=hours)

# ----------------------------
# Weather year B
# ----------------------------
wind_cf_B = pd.Series(
[0.22,0.20,0.18,0.17,0.16,0.15,0.14,0.16,0.18,0.20,0.22,0.24,
0.26,0.28,0.27,0.25,0.23,0.22,0.21,0.20,0.21,0.22,0.23,0.24],
index=hours)

solar_cf_B = pd.Series(
[0,0,0,0,0,0.05,0.18,0.35,0.55,0.72,0.82,0.88,
0.84,0.76,0.60,0.40,0.20,0.06,0,0,0,0,0,0],
index=hours)

# ----------------------------
# Model function
# ----------------------------
def run_model(wind_cf, solar_cf, label):

    n = pypsa.Network()
    n.set_snapshots(hours)

    n.add("Bus", "electricity", carrier="electricity")

    n.add("Load",
          "demand",
          bus="electricity",
          p_set=load)

    n.add("Generator",
          "wind",
          bus="electricity",
          p_max_pu=wind_cf,
          capital_cost=600,
          marginal_cost=0,
          p_nom_extendable=True)

    n.add("Generator",
          "solar",
          bus="electricity",
          p_max_pu=solar_cf,
          capital_cost=400,
          marginal_cost=0,
          p_nom_extendable=True)

    n.add("Generator",
          "gas",
          bus="electricity",
          p_max_pu=1,
          capital_cost=800,
          marginal_cost=70,
          p_nom_extendable=True)

    n.optimize()

    result = {
        "year": label,
        "avg_wind_cf": wind_cf.mean(),
        "avg_solar_cf": solar_cf.mean(),
        "wind_capacity": n.generators.at["wind","p_nom_opt"],
        "solar_capacity": n.generators.at["solar","p_nom_opt"],
        "gas_capacity": n.generators.at["gas","p_nom_opt"]
    }

    return n, result


# ----------------------------
# Run both weather years
# ----------------------------
nA, resA = run_model(wind_cf_A, solar_cf_A, "Year A")
nB, resB = run_model(wind_cf_B, solar_cf_B, "Year B")

results = pd.DataFrame([resA,resB])

print(results)

In [ ]:
import matplotlib.pyplot as plt

# ----------------------------
# Capacity factors
# ----------------------------
cf_table = results[["year","avg_wind_cf","avg_solar_cf"]].set_index("year")

cf_table.plot(kind="bar")
plt.title("Average renewable capacity factors by year")
plt.ylabel("Capacity factor")
plt.show()


# ----------------------------
# Installed capacities
# ----------------------------
capacity_table = results[
["year","wind_capacity","solar_capacity","gas_capacity"]
].set_index("year")

capacity_table.plot(kind="bar")
plt.title("Optimized installed capacities")
plt.ylabel("MW")
plt.show()


# ----------------------------
# Dispatch plots
# ----------------------------
dispatch_A = nA.generators_t.p[["wind","solar","gas"]]
dispatch_A.plot()
plt.title("Dispatch Year A")
plt.ylabel("MW")
plt.show()

dispatch_B = nB.generators_t.p[["wind","solar","gas"]]
dispatch_B.plot()
plt.title("Dispatch Year B")
plt.ylabel("MW")
plt.show()

### Part C

In [ ]:
# Code part C

In [ ]:
# Plots part C

### Part D

In [ ]:
# Code part D

In [ ]:
# Plots part D

### Part E